# Effect of Antisemitic Tropes on Recall 

In [1]:
import pandas as pd
import numpy as np
import os
import sys
from os.path import join, abspath

parent_dir = abspath(os.path.join(os.getcwd(), '../..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from config import DATA_DIR
from utils.classification_helpers import group_ihra_content, group_lexicon_content
from utils.stats_helpers import pairwise_segment_tests
from utils.analysis_helpers import calculate_recall, summarize_recall_stats, group_dfs_by_row_index_pivot, explode_df, test_recall_per_content_group_compared_to_base, combine_dataset_tests

pd.set_option("display.max_rows", 100)

PROVIDER = "gpt"

In [2]:
bloomington_tmp = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_1.feather"))
decoding = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_1.feather"))

In [3]:
print("Bloomington columns:", bloomington_tmp.columns)

Bloomington columns: Index(['comment_cleaned', 'id', 'keyword', 'ihra_section_1', 'ihra_section_2',
       'classification_tax', 'explanation_tax', 'classification_tax_ex',
       'explanation_tax_ex', 'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_no_kb_cleaned',
       'explanation_ihra_explanation_sections', 'explanation_tax_chapters',
       'explanation_tax_ex_chapters', 'ihra_sections',
       'explanation_tax_chapters_no', 'explanation_tax_ex_chapters_no',
       'explanation_tax_sections', 'explanation_tax_ex_sections'],
      dtype='object')


In [4]:
column_name_renaming = {
    'classification_no_kb_cleaned': 'NO_KB',
    'classification_ihra_explanation_cleaned': 'IHRA',
    'classification_tax': 'TAX',
    'classification_tax_ex': 'TAX_EX',
}

bloomington_tmp.rename(columns=column_name_renaming, inplace=True)
decoding.rename(columns=column_name_renaming, inplace=True)

REL_CLASS_COLS = column_name_renaming.values()

In [5]:
# Create a new df for Bloomington data with a new column "ihra_sec_new" containing both annotators' decisions.
bloomington_copy_x = bloomington_tmp.copy()
bloomington_copy_y = bloomington_tmp.copy()
bloomington_copy_x["ihra_sec_new"] = bloomington_tmp["ihra_section_1"]
bloomington_copy_y["ihra_sec_new"] = bloomington_tmp["ihra_section_2"]
bloomington = pd.concat([bloomington_copy_x, bloomington_copy_y], ignore_index=True)
print(len(bloomington))
bloomington = bloomington[bloomington["ihra_sec_new"]!=13] # there are a few errors coded as 13
print(len(bloomington))

3716
3635


## Bloomington

#### a. Within each KB

In [6]:

class_cols_to_recalls = {}
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=bloomington, classification_column=classification_column, split_by='keyword', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"------{classification_column}-------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print("\n[Pairwise tests vs each other] (Holm-corrected)")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")    


------NO_KB-------
         Recall  Support  Correct  Missed
Israel     0.12      641       79     562
Kikes      0.99       89       88       1
ZioNazi    0.91      397      361      36
Jews       0.58      689      398     291

[Pairwise tests vs each other] (Holm-corrected)
     seg_A    seg_B          p_raw          p_adj  significant  effect_h
1   Israel  ZioNazi  3.177228e-136  1.906337e-135         True -1.824724
0   Israel    Kikes   4.481685e-73   2.240843e-72         True -2.233775
2   Israel     Jews   2.311467e-66   9.245870e-66         True -1.024004
5  ZioNazi     Jews   3.911925e-30   1.173578e-29         True  0.800720
4    Kikes     Jews   1.157533e-13   2.315067e-13         True  1.209771
3    Kikes  ZioNazi   1.964614e-02   1.964614e-02         True  0.409050
---------

------IHRA-------
         Recall  Support  Correct  Missed
Israel     0.14      641       92     549
Kikes      0.97       89       86       3
ZioNazi    0.88      397      349      48
Jews       0.5

#### b. Between KBs

In [7]:
grouped = group_dfs_by_row_index_pivot(class_cols_to_recalls)
print(grouped["Kikes"])

         Correct  Missed  Recall  Support
dataset                                  
IHRA          86       3    0.97       89
NO_KB         88       1    0.99       89
TAX           87       2    0.98       89
TAX_EX        86       3    0.97       89


In [8]:
print(grouped["Israel"])

         Correct  Missed  Recall  Support
dataset                                  
IHRA          92     549    0.14      641
NO_KB         79     562    0.12      641
TAX          272     369    0.42      641
TAX_EX       311     330    0.49      641


In [9]:
# Pairwise chi-squared (auto-fallback to Fisher), Holm-corrected
model_comparison_per_group = pd.DataFrame(columns=["seg_A","seg_B", "p_raw","p_adj","significant", "effect_h"])

index = []
for i, k in enumerate(grouped.keys()):
    pairwise = pairwise_segment_tests(grouped[k], correction="holm")
    model_comparison_per_group.loc[i] = pairwise.loc[1] # change 1 to another number to get a different comparison
    index.append(k)
model_comparison_per_group["keyword"] = index
model_comparison_per_group.set_index("keyword", inplace=True)
model_comparison_per_group

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
keyword,,,,,,
Israel,IHRA,TAX,0.0,0.0,True,-0.643112
Jews,IHRA,TAX,0.001923,0.011537,True,-0.161778
Kikes,IHRA,TAX,1.0,1.0,False,-0.064372
ZioNazi,IHRA,TAX,0.000617,0.003084,True,-0.256456


### Evaluation by content groups

Content groups allow to compare IHRA sections with LEXICON chapters

In [10]:
bloomington["ihra_content"] = bloomington["ihra_sec_new"].map(group_ihra_content)
print(bloomington["ihra_content"].value_counts(dropna=False, normalize=True))
print(type(bloomington["ihra_content"][0]))

ihra_content
israel                  0.576891
classic_power           0.279230
aggressive              0.096286
second_postholocaust    0.047593
Name: proportion, dtype: float64
<class 'str'>


#### a. Within each KB

In [11]:
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=bloomington, classification_column=classification_column, split_by='ihra_content', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"--------{classification_column}--------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")    

--------NO_KB--------
                      Recall  Support  Correct  Missed
israel                  0.36     1048      374     674
classic_power           0.73      507      373     134
aggressive              0.80      175      140      35
second_postholocaust    0.47       86       40      46
           seg_A                 seg_B         p_raw         p_adj  \
0         israel         classic_power  2.658971e-44  1.595382e-43   
1         israel            aggressive  1.022829e-27  5.114143e-27   
5     aggressive  second_postholocaust  8.584257e-08  3.433703e-07   
4  classic_power  second_postholocaust  8.678374e-07  2.603512e-06   
2         israel  second_postholocaust  5.903966e-02  1.180793e-01   
3  classic_power            aggressive  1.102239e-01  1.180793e-01   

   significant  effect_h  
0         True -0.761789  
1         True -0.927295  
5         True  0.703537  
4         True  0.538031  
2        False -0.223758  
3        False -0.165506  
---------

--------IHRA

In [ ]:
kb_to_recall_b = {}
for kb in REL_CLASS_COLS:
    kb_to_recall_b[kb] = class_cols_to_recalls[kb]['Recall'].to_dict()
pd.DataFrame(kb_to_recall_b)

#### b. Between KBs

In [19]:
test_recall_per_content_group_compared_to_base(class_cols_to_recalls, 0)

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,IHRA,0.165076,0.825379,False,0.165506
classic_power,NO_KB,IHRA,0.027836,0.139181,False,0.131078
israel,NO_KB,IHRA,0.649488,0.649488,False,-0.020772
second_postholocaust,NO_KB,IHRA,1.0,1.0,False,0.040131


In [20]:
test_recall_per_content_group_compared_to_base(class_cols_to_recalls, 3)

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,TAX,1.0,1.0,False,-0.025242
classic_power,NO_KB,TAX,0.51639,1.0,False,-0.045604
israel,NO_KB,TAX,0.0,0.0,True,-0.404084
second_postholocaust,NO_KB,TAX,0.445766,1.0,False,-0.120072


In [21]:
test_recall_per_content_group_compared_to_base(class_cols_to_recalls, 4)

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,TAX_EX,0.694525,1.0,False,0.049115
classic_power,NO_KB,TAX_EX,0.723647,1.0,False,0.022397
israel,NO_KB,TAX_EX,0.0,0.0,True,-0.485152
second_postholocaust,NO_KB,TAX_EX,0.541838,1.0,False,-0.100047


In [ ]:
recall_tests_b = pd.concat([test_recall_per_content_group_compared_to_base(class_cols_to_recalls, i) for i in [0,3,4]])
recall_tests_b

## Decoding

### Evaluation by content groups



In [23]:
decoding["lexicon_chapter_groups"] = decoding["comment_codes_all_chapters"].map(group_lexicon_content)

In [24]:
# restrict to texts with maximum two content groups assigned
decoding = decoding[decoding['lexicon_chapter_groups'].map(lambda x: len(x) <=2 and len(x)>0)].copy()
decoding["lexicon_chapter_groups"] = decoding["lexicon_chapter_groups"].map(lambda x: x if len(x)==2 else x*2)
decoding = explode_df(decoding, "lexicon_chapter_groups")

In [25]:
decoding["lexicon_chapter_groups"].value_counts()

lexicon_chapter_groups
israel                  5334
classic_power           4338
second_postholocaust    1550
aggressive               262
Name: count, dtype: int64

#### a. Within each KB

In [26]:
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=decoding, classification_column=classification_column, split_by='lexicon_chapter_groups', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"--------{classification_column}--------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print("\n[Pairwise tests vs each other] (Holm-corrected)")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")   

--------NO_KB--------
                      Recall  Support  Correct  Missed
israel                  0.19     2667      502    2165
classic_power           0.31     2169      680    1489
second_postholocaust    0.24      775      186     589
aggressive              0.53      131       70      61

[Pairwise tests vs each other] (Holm-corrected)
                  seg_A                 seg_B         p_raw         p_adj  \
0                israel         classic_power  9.267315e-24  5.560389e-23   
2                israel            aggressive  2.552928e-21  1.276464e-20   
5  second_postholocaust            aggressive  9.385206e-12  3.754082e-11   
4         classic_power            aggressive  2.743994e-07  8.231981e-07   
3         classic_power  second_postholocaust  1.395799e-04  2.791598e-04   
1                israel  second_postholocaust  1.799551e-03  1.799551e-03   

   significant  effect_h  
0         True -0.278946  
2         True -0.728779  
5         True -0.606887  
4     

In [ ]:
kb_to_recall_d = {}
for kb in REL_CLASS_COLS:
    kb_to_recall_d[kb] = class_cols_to_recalls[kb]['Recall'].to_dict()
pd.DataFrame(kb_to_recall_d)

#### b. Between KBs

In [ ]:
recall_tests_d = pd.concat([test_recall_per_content_group_compared_to_base(class_cols_to_recalls, i) for i in [0,3,4]])
recall_tests_d

## Dataset union 

- equal dataset weight
- by content groups

In [33]:
# combine recalls per content group
# also need to fetch overall recall first

In [34]:
kb_to_recall_bd = {}
for classification_column in REL_CLASS_COLS:
    tmp_b = kb_to_recall_b[classification_column]
    tmp_d = kb_to_recall_d[classification_column]
    tmp = {k:0.5*(tmp_b[k] + tmp_d[k]) for k in tmp_b.keys()}
    kb_to_recall_bd[classification_column] = tmp
       

In [35]:
kb_to_recall_bd = pd.DataFrame(kb_to_recall_bd)
kb_to_recall_bd

,NO_KB,IHRA,TAX,TAX_EX
israel,0.275,0.280,0.545,0.605
classic_power,0.520,0.450,0.585,0.595
aggressive,0.665,0.565,0.735,0.760
second_postholocaust,0.355,0.300,0.460,0.490


In [36]:
kb_to_recall_bd.loc["mean",:] = kb_to_recall_bd.mean().tolist()
kb_to_recall_bd

,NO_KB,IHRA,TAX,TAX_EX
israel,0.27500,0.28000,0.54500,0.6050
classic_power,0.52000,0.45000,0.58500,0.5950
aggressive,0.66500,0.56500,0.73500,0.7600
second_postholocaust,0.35500,0.30000,0.46000,0.4900
mean,0.45375,0.39875,0.58125,0.6125


In [28]:
kb_to_recall_bd.to_parquet(f"{PROVIDER}_recall_by_trope.parquet", index=True)


In [37]:
combined = combine_dataset_tests(
    recall_tests_b,
    recall_tests_d,
)
combined

,content_group,seg_A,seg_B,p_raw,effect_h,p_adj,significant
6,israel,NO_KB,TAX,7.274069e-30,-0.566431,8.728883e-29,True
10,israel,NO_KB,TAX_EX,7.274069e-30,-0.687855,8.728883e-29,True
11,second_postholocaust,NO_KB,TAX_EX,1.013900e-09,-0.283406,1.013900e-08,True
1,classic_power,NO_KB,IHRA,3.133558e-09,0.155860,2.795817e-08,True
9,classic_power,NO_KB,TAX_EX,3.106464e-09,-0.153682,2.795817e-08,True
5,classic_power,NO_KB,TAX,6.312163e-08,-0.137355,4.418514e-07,True
7,second_postholocaust,NO_KB,TAX,4.223115e-07,-0.222554,2.533869e-06,True
3,second_postholocaust,NO_KB,IHRA,2.440615e-03,0.134339,1.220307e-02,True
8,aggressive,NO_KB,TAX_EX,8.283526e-03,-0.195752,3.313410e-02,True
0,aggressive,NO_KB,IHRA,1.349541e-02,0.213450,4.048623e-02,True
